# AgentCore Runtime with no framework at all

**The claim this notebook tests:** AgentCore Runtime hosts a container that answers `POST /invocations` and `GET /ping`. It does not know or care whether Strands, LangGraph, or a hand-written loop is inside.

Same four commands as notebook 1. Different code inside the box.

```mermaid
flowchart LR
    subgraph SAME[Identical across every framework]
        C[agentcore create] --> D[agentcore dev]
        D --> P[agentcore deploy]
        P --> I[agentcore invoke]
    end
    subgraph DIFF[The only part that changes]
        X[main.py: your agent loop]
    end
    X --> SAME
```

**Do you need a second project?** Yes, and the reason is blast radius.

| Option | What happens | Pick it when |
|---|---|---|
| A second project (this notebook) | Its own CDK stack. A broken agent cannot take down the working one | Teaching, demos, anything you run live |
| `agentcore add agent --name PlainAgent` in the existing project | One stack, one `deploy` for both. A dependency failure in either fails the whole stack | A real product where the agents ship together |

Running a live session, take the isolated project. One failing build should not black out a demo that already worked.

In [ ]:
import json, os, re, subprocess, uuid, pathlib
import boto3

REGION   = "us-east-1"
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

def sh(cmd, cwd=None, timeout=900, quiet=False):
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    if not quiet:
        print(f"$ {cmd}")
        print(out[:4000] if out else "(no output)")
        print("-" * 70)
    return p.returncode, out

# Where the new project goes. Same parent folder as the Strands project by default.
WORKDIR = pathlib.Path(
    "/workspace/"
    "Day-11 - AgentCore/demos"
).expanduser()

PROJECT_NAME = "PlainRuntimeAgent"
AGENT_NAME   = "PlainAgent"
PROJECT_DIR  = WORKDIR / PROJECT_NAME
AGENT_DIR    = PROJECT_DIR / "app" / AGENT_NAME

print("workdir :", WORKDIR, "exists" if WORKDIR.exists() else "MISSING")
print("project :", PROJECT_DIR, "exists" if PROJECT_DIR.exists() else "not created yet")
sh("agentcore --version", quiet=True)

## 1. Create the project

The CLI has no "no framework" option, so scaffold with Strands and then delete it. That is the point being made: the scaffold is convenience, the contract is HTTP.

| Flag | Value | Why |
|---|---|---|
| `--framework` | `Strands` | Only to get a code-based project instead of a harness. We overwrite the code |
| `--protocol` | `HTTP` | The `/invocations` contract |
| `--build` | `CodeZip` | No Docker needed |
| `--memory` | `none` | Zero extra AWS resources |
| `--model-provider` | `Bedrock` | AWS credentials, no API key file |

In [ ]:
# Idempotent: skips creation if the folder already exists.
if PROJECT_DIR.exists():
    print("project already exists, skipping create")
else:
    cmd = (
        "agentcore create "
        f"--project-name {PROJECT_NAME} "
        f"--name {AGENT_NAME} "
        "--language Python --framework Strands --model-provider Bedrock "
        "--protocol HTTP --build CodeZip --memory none"
    )
    sh(cmd, cwd=str(WORKDIR), timeout=1800)

cfg_path = PROJECT_DIR / "agentcore" / "agentcore.json"
cfg = json.loads(cfg_path.read_text()) if cfg_path.exists() else {}
RT = (cfg.get("runtimes") or [{}])[0]
ENTRYPOINT = RT.get("entrypoint", "main.py")
AGENT_DIR  = (PROJECT_DIR / RT.get("codeLocation", f"app/{AGENT_NAME}")).resolve()
AGENT_FILE = AGENT_DIR / ENTRYPOINT.split(":")[0]

print("entrypoint :", AGENT_FILE)
print("module     :", AGENT_FILE.stem + ":app")
tgt = PROJECT_DIR / "agentcore" / "aws-targets.json"
print("targets    :", tgt.read_text().strip()[:300] if tgt.exists() else "missing")

## 2. The agent loop, written by hand

No framework means you own the loop. It is smaller than people expect: call the model, check whether it asked for a tool, run the tool, feed the result back, repeat until it stops asking.

```mermaid
flowchart TD
    S[user prompt] --> C[converse: messages + toolConfig]
    C --> D{stopReason}
    D -- end_turn --> A[return the text]
    D -- tool_use --> T[run the local function]
    T --> R[append toolResult as a user message]
    R --> C
    D -- iteration cap hit --> X[stop and say so]
```

| Design choice | Why it matters |
|---|---|
| A hard iteration cap | Without it, a confused model can loop until the request times out and the bill is real |
| Append the assistant message **before** the tool result | Bedrock rejects a `toolResult` that does not follow the matching `toolUse` |
| Tool errors returned as content, not raised | The model can recover from "PNR not found". It cannot recover from a 500 |
| No `top_p` alongside `temperature` | Claude 4.x treats them as mutually exclusive |

In [ ]:
AGENT_SRC = '''"""Minimum AgentCore Runtime agent with no agent framework.

Amazon Bedrock Converse API plus a hand written tool loop.
Contract: POST /invocations with {"prompt": "..."} -> {"result": "...", "turns": n}
"""

import json
import os

import boto3
from bedrock_agentcore.runtime import BedrockAgentCoreApp

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
REGION = os.environ.get("AWS_REGION", os.environ.get("AWS_DEFAULT_REGION", "us-east-1"))
MAX_TURNS = 5

SYSTEM = [{"text": (
    "You are TravelMind, an airline support agent. "
    "Use get_pnr for any question about a booking. "
    "Answer in three sentences or fewer. Never invent a booking."
)}]

TOOL_CONFIG = {"tools": [{"toolSpec": {
    "name": "get_pnr",
    "description": "Look up an airline booking by its PNR code.",
    "inputSchema": {"json": {
        "type": "object",
        "properties": {"pnr": {"type": "string", "description": "Six character PNR code"}},
        "required": ["pnr"],
    }},
}}]}

_BOOKINGS = {
    "JX48Q2": {"passenger": "Rao", "tier": "Gold",
               "segment": "BLR-DEL", "status": "CANCELLED"},
}

app = BedrockAgentCoreApp()
bedrock = boto3.client("bedrock-runtime", region_name=REGION)


def get_pnr(pnr: str) -> dict:
    """The tool implementation. Returns data, never raises at the model."""
    return _BOOKINGS.get(str(pnr).strip().upper(), {"error": "PNR not found"})


TOOLS = {"get_pnr": get_pnr}


def run_loop(prompt: str) -> dict:
    """Call the model, service tool requests, stop at end_turn or the cap."""
    messages = [{"role": "user", "content": [{"text": prompt}]}]

    for turn in range(1, MAX_TURNS + 1):
        response = bedrock.converse(
            modelId=MODEL_ID,
            messages=messages,
            system=SYSTEM,
            toolConfig=TOOL_CONFIG,
            inferenceConfig={"maxTokens": 512, "temperature": 0.2},
        )
        out_message = response["output"]["message"]
        messages.append(out_message)               # assistant turn must be appended first

        if response.get("stopReason") != "tool_use":
            text = "".join(b.get("text", "") for b in out_message["content"])
            return {"result": text.strip(), "turns": turn}

        tool_results = []
        for block in out_message["content"]:
            use = block.get("toolUse")
            if not use:
                continue
            fn = TOOLS.get(use["name"])
            payload = fn(**use["input"]) if fn else {"error": f"unknown tool {use['name']}"}
            tool_results.append({"toolResult": {
                "toolUseId": use["toolUseId"],
                "content": [{"json": payload}],
                "status": "error" if "error" in payload else "success",
            }})

        messages.append({"role": "user", "content": tool_results})

    return {"result": "Stopped after the maximum number of turns.", "turns": MAX_TURNS}


@app.entrypoint
def invoke(payload, context):
    prompt = payload.get("prompt") if isinstance(payload, dict) else None
    if not isinstance(prompt, str) or not prompt.strip():
        return {"error": "payload must contain a non-empty 'prompt' string"}

    out = run_loop(prompt)
    out["session_id"] = getattr(context, "session_id", None)
    return out


if __name__ == "__main__":
    app.run()
'''

AGENT_DIR.mkdir(parents=True, exist_ok=True)
AGENT_FILE.write_text(AGENT_SRC)

# The scaffold leaves model/ and mcp_client/ behind. Neither is imported now.
for leftover in ("model", "mcp_client", "skills"):
    d = AGENT_DIR / leftover
    if d.exists():
        print("scaffold leftover, unused by this agent:", d)

print("wrote", AGENT_FILE, f"({len(AGENT_SRC)} bytes)")

In [ ]:
# Dependencies: no strands, no mcp. Compare this with notebook 1.
PYPROJECT_SRC = f'''[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "{AGENT_NAME.lower()}"
version = "0.1.0"
description = "AgentCore Runtime application with no agent framework"
requires-python = ">=3.10"
dependencies = [
    "aws-opentelemetry-distro",
    "bedrock-agentcore>=1.9.1",
    "boto3>=1.40.0",
    "botocore[crt]>=1.35.0",
]

[tool.hatch.build.targets.wheel]
packages = ["."]
'''

pyproject = AGENT_DIR / "pyproject.toml"
pyproject.write_text(PYPROJECT_SRC)
print(PYPROJECT_SRC)

code, _ = sh("uv sync", cwd=str(AGENT_DIR), timeout=900)
print("uv sync exit code:", code, "(non-zero means the dev server would start on a broken venv)")

In [ ]:
# In-process test through the agent's own venv.
venv_py = AGENT_DIR / ".venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
print("venv python:", "found" if venv_py.exists() else "MISSING")

harness = f'''
import json, sys
sys.path.insert(0, {str(AGENT_DIR)!r})
from {AGENT_FILE.stem} import invoke

class Ctx:
    session_id = "local-test-" + "0" * 24

print(json.dumps(invoke({{"prompt": "What is the status of PNR JX48Q2?"}}, Ctx()), indent=2))
print(json.dumps(invoke({{"prompt": "What about PNR ZZ0000?"}}, Ctx()), indent=2))
print(invoke({{}}, Ctx()))
'''
hp = AGENT_DIR / "_local_test.py"
hp.write_text(harness)
if venv_py.exists():
    sh(f'"{venv_py}" "{hp}"', cwd=str(AGENT_DIR), timeout=300)
hp.unlink(missing_ok=True)

Three test prompts, three different paths through the loop:

| Prompt | Expected path | What it proves |
|---|---|---|
| PNR JX48Q2 | `tool_use` then `end_turn`, 2 turns | The tool loop works |
| PNR ZZ0000 | Tool returns an error payload, model recovers | Failure is data, not an exception |
| empty payload | Guard fires before any model call | Input validation costs nothing and saves tokens |

## 3. Local server

Terminal A, from the project root:

```bash
cd "<path>/PlainRuntimeAgent"
agentcore dev
```

Terminal B:

```bash
agentcore dev "What is the status of PNR JX48Q2?"
curl http://localhost:8080/ping
```

If notebook 1's dev server is still running on 8080, use `agentcore dev -p 8081` here. Two projects, two ports.

In [ ]:
# Deploy. Run `agentcore deploy` once in a terminal first if CDK bootstrap is needed.
code, out = sh("agentcore deploy -y", cwd=str(PROJECT_DIR), timeout=3600)
print("deploy exit code:", code)

In [ ]:
code, status_out = sh("agentcore status --json", cwd=str(PROJECT_DIR), timeout=300, quiet=True)
arns = sorted(set(re.findall(r"arn:aws[\w-]*:bedrock-agentcore:[^\"'\s,]+runtime/[^\"'\s,]+", status_out)))
AGENT_ARN = arns[0] if arns else None
print("runtime ARN:", AGENT_ARN or "not deployed yet")

SESSION_ID = "plain-demo-" + uuid.uuid4().hex
sh(f'agentcore invoke --prompt "Status of PNR JX48Q2?" --session-id {SESSION_ID}',
   cwd=str(PROJECT_DIR), timeout=600)

In [ ]:
# Same boto3 client as notebook 1. Nothing about the caller changes.
acr = boto3.client("bedrock-agentcore", region_name=REGION)

def ask(prompt: str, session_id: str, arn: str = None):
    resp = acr.invoke_agent_runtime(
        agentRuntimeArn=arn or AGENT_ARN,
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode("utf-8"),
        qualifier="DEFAULT",
    )
    raw = b"".join(chunk for chunk in resp.get("response", []))
    text = raw.decode("utf-8", errors="replace").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"raw": text}

if AGENT_ARN:
    print(json.dumps(ask("Status of PNR JX48Q2?", SESSION_ID), indent=2)[:1200])
else:
    print("Deploy first.")

## 4. The comparison that makes the point

Two agents, two frameworks, one platform. Line up what changed against what did not.

| | Notebook 1 (Strands) | Notebook 2 (no framework) |
|---|---|---|
| Agent loop | `Agent(...)` handles it | ~40 lines you wrote and own |
| Tool definition | `@tool` plus a docstring | A `toolSpec` JSON schema by hand |
| Tool error handling | Contained as `status="error"` | You build the `toolResult` yourself |
| Iteration cap | Framework default | Your `MAX_TURNS` |
| Extra dependency | `strands-agents` | none |
| Entrypoint decorator | `@app.entrypoint` | `@app.entrypoint` |
| Project layout | identical | identical |
| Commands | identical | identical |
| ARN, session id, invoke path | identical | identical |
| Logs, traces, IAM role | identical | identical |

**The takeaway for an architecture conversation:** the framework is a productivity decision, reversible in one file. The platform is an operations decision that touches IAM, networking, observability, and cost. Do not let a framework debate stand in for a platform decision, and do not let a framework choice quietly lock the platform.

**Skeptic's question worth putting to the room:** if the hand-written loop is only forty lines, what exactly is the framework earning? Fair answers: session and conversation management, streaming event shapes, retries, multi-agent handoff, tool executors, hooks, and a tool ecosystem you would otherwise write. Unfair answer: "everyone uses it."

## 5. Cleanup

```bash
cd "<path>/PlainRuntimeAgent"
agentcore remove all
agentcore deploy
```

Two projects means two teardowns. Check both, or check the CloudFormation console for any stack named after either project.

**Next:** the same loop with LangChain and LangGraph. The `main.py` changes, `pyproject.toml` gains `langchain` and `langgraph`, and everything from `agentcore create` to `invoke_agent_runtime` stays exactly where it is.